<a href="https://colab.research.google.com/github/29hashir04/Text_to_video/blob/main/Script_To_AI_Video_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q anthropic fal-client aiohttp pydantic nest_asyncio

In [ ]:
!pip install -q gradio_client requests Pillow ipywidgets

In [ ]:
import os
import json
import re
import time
import random
import shutil
import urllib.parse
import subprocess
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
from gradio_client import Client, handle_file
from google.colab import files

# ==========================================
# SYSTEM SOPs & CLIENT CHECKLIST
# ==========================================
SOP_PROMPT = """
You are an Anime Commercial Director (Ufotable / Makoto Shinkai style).
Break the user's script into sequential 3-second shots.

STRICT CLIENT SOPs:
1. NO EARS, NO EYES, NO NOSE visible on any character from any angle.
2. Every character MUST wear pitch-black sunglasses, headwear (hoodie pulled up, kufi, beanie, cap, or ghutrah) completely covering ears and hair, and a full beard covering the lower nose and jawline.
3. Rotate camera angles on every shot (Low-Angle Hero Shot, Over-The-Shoulder, Bird's-Eye View, Extreme Macro POV on hands/desk, Wide Rear Tracking).
4. Alternate ZOOM_DIR between zoom_in and zoom_out on every shot.
5. Set IS_PAST to TRUE only when the line discusses the past or initial struggle (triggers B&W filter), otherwise FALSE.
6. Logical real-world actions only. Different clothes per character and location. No women.

OUTPUT FORMAT RULES:
Do NOT output JSON or markdown. Output strictly ONE LINE PER SHOT separated by the pipe character `|` with exactly 6 fields per line:
CAMERA_ANGLE | ZOOM_DIR | IS_PAST | SCRIPT_LINE | IMAGE_PROMPT | MOTION_PROMPT
"""

# ==========================================
# OFFLINE DETERMINISTIC DIRECTOR (CAN NEVER FAIL OR GIVE 402)
# ==========================================
CAMERA_ROTATION = [
    ("Over-The-Shoulder (OTS) Medium Shot", "Minimalist luxury dark executive office with floor-to-ceiling glass city view"),
    ("High-Angle Bird's-Eye 45-Degree Shot", "Modern tech agency workstation with glowing dual monitors showing green analytics"),
    ("Extreme Macro Top-Down POV Shot", "Close-up on desk surface, hands with steel wristwatch typing on mechanical keyboard beside steaming black coffee"),
    ("Side-Profile Dutch Angle Shot", "Tech cubicle workstation with clean code on monitor and upward green sales graph"),
    ("Low-Angle Hero Shot (Looking Up)", "High-rise glass terrace at golden hour sunset with dramatic violet and amber clouds"),
    ("Wide Rear-View Tracking Shot", "Sunlit white marble archway courtyard in Medina with palm tree shadows"),
    ("Eye-Level 45-Degree Medium Shot", "Creative urban warehouse studio with branded apparel packaging on wooden table"),
    ("Low-Angle Diagonal Boardroom Shot", "Modern glass conference room at night with digital revenue curve on screen"),
    ("Ultra-Wide Establishing Drone Shot", "Modern global logistics port at sunrise with automated cranes and cargo ship"),
    ("Rear High-Angle Auditorium Shot", "Grand modern conference hall looking from behind seated male audience toward illuminated stage"),
    ("Over-The-Shoulder Tight Tech POV", "Dual curved monitors showing backend code and incoming payout notification"),
    ("Eye-Level Respectful Library Shot", "Classical Islamic study with dark mahogany bookshelves and warm golden lamplight")
]

WARDROBE_ROTATION = [
    "wearing charcoal thobe, flat cap covering ears completely, pitch-black sunglasses hiding eyes, thick dark beard covering nose and jaw",
    "wearing dark hoodie pulled up covering ears completely, pitch-black vision glasses hiding eyes, neat full beard covering nose",
    "wearing olive bomber jacket, black beanie pulled low over ears, pitch-black sunglasses hiding eyes, trimmed beard covering nose",
    "wearing white linen thobe and keffiyeh wrapped covering head and ears completely, pitch-black sunglasses hiding eyes, full beard",
    "wearing tailored navy blazer, black kufi covering ears completely, dark round pitch-black sunglasses hiding eyes, sharp beard",
    "wearing traditional brown gold-trimmed bisht and white ghutrah draped over ears, pitch-black sunglasses hiding eyes, full silver beard"
]

def local_deterministic_director(script_text: str) -> list:
    """Splits script into 3s sentences and builds compliant anime prompts offline in Python."""
    raw_sentences = re.split(r'(?<=[.!?])\s+|\n+', script_text.strip())
    sentences = [s.strip() for s in raw_sentences if len(s.strip()) > 3]

    shots = []
    for i, sent in enumerate(sentences):
        shot_num = i + 1
        cam_angle, env_desc = CAMERA_ROTATION[i % len(CAMERA_ROTATION)]
        wardrobe = WARDROBE_ROTATION[i % len(WARDROBE_ROTATION)]
        zoom_dir = "zoom_in" if (shot_num % 2 == 1) else "zoom_out"

        # Check if sentence discusses the past / initial struggle
        lower_s = sent.lower()
        is_past = any(p in lower_s for p in [
            "found abu", "nobody else had said", "over the past", "used to",
            "before", "back then", "keep his job", "years ago"
        ])
        if any(p in lower_s for p in ["now making", "15k profit", "16.7k", "runs the biggest"]):
            is_past = False

        if "Macro" in cam_angle or "Drone" in cam_angle:
            img_prompt = f"{cam_angle} of {env_desc}, crisp Ufotable anime cel-shading, dramatic lighting, no faces visible"
        else:
            img_prompt = (
                f"{cam_angle} in {env_desc}, Muslim man {wardrobe}, "
                f"strictly no eyes visible, no ears visible, no nose visible, dramatic rim lighting"
            )

        mot_prompt = (
            f"Progressive smooth {zoom_dir.replace('_', ' ')} camera movement, "
            f"subtle wind and ambient lighting physics, 24fps anime pacing"
        )

        shots.append({
            "shot_number": shot_num,
            "camera_angle": cam_angle,
            "zoom_direction": zoom_dir,
            "is_past": is_past,
            "script_line": sent,
            "image_prompt": img_prompt,
            "motion_prompt": mot_prompt
        })
    return shots

# ==========================================
# STAGE 1: SCRIPT -> TEXT PROMPTS (MULTI-FALLBACK)
# ==========================================
def step1_process_script_to_text(script_text: str, out_dir: str):
    print("\n[STAGE 1/3] Processing Script & Client SOPs into Text Prompts...")
    shots = []

    # Try Free HuggingFace Qwen2.5 Space first (No API key, no 402 error)
    try:
        print(" -> Connecting to Free HuggingFace LLM Space (Qwen2.5-72B)...")
        llm_client = Client("Qwen/Qwen2.5-72B-Instruct")
        result = llm_client.predict(
            query=f"{SOP_PROMPT}\n\nBreak this entire script into 3-second shots using the pipe `|` format:\n\n{script_text}",
            history=[],
            system=SOP_PROMPT,
            api_name="/model_chat"
        )
        raw_reply = result[1][0][1] if isinstance(result, tuple) else str(result)

        shot_counter = 1
        for line in raw_reply.splitlines():
            if "|" not in line or "CAMERA_ANGLE" in line:
                continue
            parts = [p.strip() for p in line.split("|")]
            if len(parts) >= 6:
                is_past = "true" in parts[2].lower() or any(w in parts[3].lower() for w in ["over the past", "found abu"])
                shots.append({
                    "shot_number": shot_counter,
                    "camera_angle": parts[0],
                    "zoom_direction": "zoom_out" if "out" in parts[1].lower() else "zoom_in",
                    "is_past": is_past,
                    "script_line": parts[3],
                    "image_prompt": parts[4],
                    "motion_prompt": parts[5]
                })
                shot_counter += 1
    except Exception as e:
        print(" -> External LLM busy; using built-in Deterministic Anime SOP Director Engine...")

    # If external LLM returned fewer than 2 shots, use deterministic engine
    if len(shots) < 2:
        shots = local_deterministic_director(script_text)

    # Save JSON & readable TXT file
    with open(f"{out_dir}/00_STORYBOARD_PROMPTS.json", "w") as f:
        json.dump({"shots": shots}, f, indent=2)

    txt_report = []
    print(f"\n--- GENERATED {len(shots)} SHOTS (SCENE BREAKDOWN) ---")
    for s in shots:
        bw_label = "BLACK & WHITE (Past)" if s["is_past"] else "FULL COLOR (Present)"
        block = (
            f"SHOT {s['shot_number']:02d} | {s['camera_angle']} | {s['zoom_direction'].upper()} | {bw_label}\n"
            f"  Script Line   : {s['script_line']}\n"
            f"  Image Prompt  : {s['image_prompt']}\n"
            f"  Motion Prompt : {s['motion_prompt']}\n"
        )
        print(block)
        txt_report.append(block)

    with open(f"{out_dir}/00_STORYBOARD_PROMPTS.txt", "w") as f:
        f.write("\n".join(txt_report))

    return shots


# ==========================================
# STAGE 2: TEXT -> IMAGES (MULTI-SERVER + FALLBACK)
# ==========================================
def step2_text_to_images(shots: list, img_dir: str):
    print("\n[STAGE 2/3] Generating Anime Keyframe Images ($0 Free Pipeline)...")

    # Connect to free HuggingFace Flux Space as primary/backup
    hf_img_client = None
    try:
        hf_img_client = Client("black-forest-labs/FLUX.1-schnell")
    except Exception:
        pass

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    for shot in shots:
        num = f"{shot['shot_number']:02d}"
        full_prompt = (
            f"Masterpiece Ufotable anime style, crisp cel-shaded line art, cinematic lighting, "
            f"{shot['camera_angle']}, {shot['image_prompt']}, "
            f"pitch black sunglasses hiding eyes completely, hat or hood hiding ears completely, thick beard hiding nose, no eyes, no ears"
        )
        img_path = f"{img_dir}/shot_{num}.png"
        print(f" -> Generating Image {num}: {shot['camera_angle']}...")
        saved = False

        # Method A: Free HuggingFace Flux.1-Schnell Space
        if hf_img_client and not saved:
            try:
                res = hf_img_client.predict(
                    prompt=full_prompt,
                    seed=shot['shot_number'] * 42,
                    randomize_seed=False,
                    width=1280,
                    height=720,
                    num_inference_steps=4,
                    api_name="/infer"
                )
                out_file = res[0] if isinstance(res, tuple) else res
                shutil.copy(out_file, img_path)
                saved = True
            except Exception:
                pass

        # Method B: Pollinations GET with Browser User-Agent
        if not saved:
            encoded = urllib.parse.quote(full_prompt)
            url = f"https://image.pollinations.ai/prompt/{encoded}?width=1280&height=720&nologo=true&seed={shot['shot_number']*42}"
            for attempt in range(3):
                try:
                    resp = requests.get(url, headers=headers, timeout=60)
                    if resp.status_code == 200 and len(resp.content) > 5000:
                        with open(img_path, "wb") as f:
                            f.write(resp.content)
                        saved = True
                        break
                except Exception:
                    time.sleep(2)

        # Method C: Guaranteed Local Pilgrim Slate if both internet servers are down
        if not saved:
            from PIL import Image, ImageDraw
            img = Image.new("RGB", (1280, 720), color=(22, 26, 35))
            d = ImageDraw.Draw(img)
            d.text((60, 320), f"Shot {num}: {shot['camera_angle']}\n{shot['script_line'][:70]}", fill=(57, 211, 83))
            img.save(img_path)

        shot["image_path"] = img_path
        time.sleep(1.5)


# ==========================================
# STAGE 3: IMAGES -> 3s VIDEOS (FFMPEG + HF)
# ==========================================
def render_final_3s_clip(source_path: str, is_image: bool, output_path: str, zoom_dir: str, is_bw: bool):
    if zoom_dir == "zoom_in":
        zoom_filter = "zoompan=z='min(zoom+0.0020,1.15)':d=72:x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s=1280x720:fps=24"
    else:
        zoom_filter = "zoompan=z='if(eq(on,1),1.15,max(1.0,zoom-0.0020))':d=72:x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s=1280x720:fps=24"

    vf_chain = [zoom_filter, "noise=alls=4:allf=t+u"]
    if is_bw:
        vf_chain.append("hue=s=0")

    cmd = ["ffmpeg", "-y"]
    if is_image:
        cmd += ["-loop", "1", "-i", source_path]
    else:
        cmd += ["-i", source_path]

    cmd += [
        "-t", "3.0",
        "-vf", ",".join(vf_chain),
        "-c:v", "libx264", "-preset", "fast", "-crf", "18",
        "-pix_fmt", "yuv420p",
        output_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def step3_images_to_videos(shots: list, raw_vid_dir: str, out_dir: str):
    print("\n[STAGE 3/3] Converting Images into 3-Second Video Clips...")
    hf_client = None
    try:
        hf_client = Client("Lightricks/LTX-Video")
    except Exception:
        print(" -> Using fast local 2.5D Anime Camera Engine for 3s clips.")

    for shot in shots:
        num = f"{shot['shot_number']:02d}"
        img_path = shot["image_path"]
        raw_vid_path = f"{raw_vid_dir}/raw_{num}.mp4"
        used_ai_video = False

        if hf_client:
            try:
                print(f" -> Animating Shot {num} via Free HuggingFace ZeroGPU...")
                result = hf_client.predict(
                    prompt=shot["motion_prompt"],
                    input_image_filepath=handle_file(img_path),
                    height_ui=512,
                    width_ui=768,
                    num_frames_ui=73,
                    api_name="/image_to_video"
                )
                video_out = result[0]["video"] if isinstance(result, tuple) else result["video"]
                shutil.copy(video_out, raw_vid_path)
                used_ai_video = True
            except Exception:
                print(f" -> Shot {num}: Applying FFmpeg 2.5D Anime Camera Motion...")

        bw_tag = "BW_" if shot.get("is_past") else ""
        final_vid_path = f"{out_dir}/{num}_shot_{bw_tag}{shot['zoom_direction']}.mp4"

        render_final_3s_clip(
            source_path=raw_vid_path if used_ai_video else img_path,
            is_image=not used_ai_video,
            output_path=final_vid_path,
            zoom_dir=shot["zoom_direction"],
            is_bw=bool(shot.get("is_past"))
        )
        print(f"    [DONE] Saved: {final_vid_path}")


# ==========================================
# INTERACTIVE UI PROMPT
# ==========================================
script_box = widgets.Textarea(
    value="",
    placeholder="Paste the client's full multi-paragraph script here...",
    description="Client Script:",
    layout=widgets.Layout(width="95%", height="200px")
)

start_btn = widgets.Button(
    description="Start Processing Pipeline ($0 Unbreakable)",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="340px", height="45px")
)

output_log = widgets.Output()

def on_start_clicked(b):
    with output_log:
        clear_output()
        user_script = script_box.value.strip()
        if not user_script:
            print("ERROR: Please paste a script into the box first!")
            return

        start_btn.disabled = True
        start_btn.description = "Processing..."

        OUT_DIR = "final_3s_clips"
        IMG_DIR = f"{OUT_DIR}/generated_images"
        RAW_VID_DIR = "temp_raw_videos"
        for d in [OUT_DIR, IMG_DIR, RAW_VID_DIR]:
            shutil.rmtree(d, ignore_errors=True)
            os.makedirs(d, exist_ok=True)

        try:
            shots = step1_process_script_to_text(user_script, OUT_DIR)
            step2_text_to_images(shots, IMG_DIR)
            step3_images_to_videos(shots, RAW_VID_DIR, OUT_DIR)

            print("\n=== ALL STAGES COMPLETE! Zipping & Downloading Everything ===")
            shutil.make_archive("final_3s_clips", "zip", OUT_DIR)
            files.download("final_3s_clips.zip")
        finally:
            start_btn.disabled = False
            start_btn.description = "Start Processing Pipeline ($0 Unbreakable)"

start_btn.on_click(on_start_clicked)

print("100% FREE & CRASH-PROOF! Paste your client's script below and click the green button:")
display(script_box, start_btn, output_log)

100% FREE & CRASH-PROOF! Paste your client's script below and click the green button:


Textarea(value='', description='Client Script:', layout=Layout(height='200px', width='95%'), placeholder="Past…

Button(button_style='success', description='Start Processing Pipeline ($0 Unbreakable)', icon='play', layout=L…

Output()